## Interactive audio sample cleaning

In [53]:
from __future__ import annotations  #Not needed for Python 3.10+
import yaml
from pathlib import Path
import matplotlib
import pandas as pd
from IPython.display import display #, HTML
import ipywidgets as widgets
button = widgets.Button(description="Continue")
output = widgets.Output()
import contextily as cx

from anqa.annotation import (MiniBirdNamer, FastMap, AnnotationState, load_labels,
                             normalize_secondary_labels, create_class_widgets,
                             SpectrogramAnnotator, AnnotationSession,
                             AnnotationControls, load_current_sample)

%matplotlib widget  
print(f'The Matplotlib backend is {matplotlib.get_backend()}')

The Matplotlib backend is widget


In [54]:
project_root = str(Path().resolve().parent)

geographic_extents = {'new_zealand': {'min_longitude': 166,
                                        'max_longitude': 179,
                                        'min_latitude': -49,
                                        'max_latitude': -34},
                      'south_america': {'min_longitude': -82,
                                        'max_longitude': -34.0,
                                        'min_latitude': -56.0,
                                        'max_latitude': 13.0}
                      }

### This cell requires manual setup

In [55]:
try:
    yaml_path = Path(project_root) / 'config.yaml'
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
except Exception as e:
    print("⚠️ There's a problem with config.yaml — check the file for typos or missing colons.")
    print(f"Details: {e}")
    raise

source_dataset_path = cfg.get("source_dataset_path") or project_root + '/data/default_source'

use_case = {
    'project_root': project_root,
    'source_dataset_path': source_dataset_path,
    'reviewed_data_destn': cfg.get("reviewed_data_destn") or project_root + '/data/default_reviewed',
    'naming_csv': cfg.get("naming_csv") or project_root + '/data/bird_names/bird_names.csv',
    'audio_folder': Path(source_dataset_path) / 'audio',
    'author': cfg["author"],
    'reviewer': cfg["reviewer"],
    'map_extents': geographic_extents[cfg["map_extents"]],
    'display_width': cfg["display_width"],
}

load_samples = False
print(use_case)

{'project_root': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa', 'source_dataset_path': 'C:/Users/ollyp/OneDrive/Desktop/anqa/data/8kHz_source', 'reviewed_data_destn': 'C:/Users/ollyp/OneDrive/Desktop/anqa/data/8khz_reviewed', 'naming_csv': 'C:/Users/ollyp/OneDrive/Desktop/anqa/data/bird_names/bird_names.csv', 'audio_folder': WindowsPath('C:/Users/ollyp/OneDrive/Desktop/anqa/data/8kHz_source/audio'), 'author': 'Olly', 'reviewer': None, 'map_extents': {'min_longitude': 166, 'max_longitude': 179, 'min_latitude': -49, 'max_latitude': -34}, 'display_width': 16}


In [56]:
class FilePaths:
    def __init__(self, options: dict):
        _project_dir = Path(options['project_root'])
        self.original_dataset = Path(options['source_dataset_path'])
        self.data_folder = _project_dir / 'data'
        self.audio_folder = options.get('audio_folder', self.data_folder)
        _metadata_files = [f for ext in ('csv', 'parquet') for f in self.original_dataset.glob(f'metadata.{ext}')]
        _label_files = [f for ext in ('csv', 'parquet') for f in self.original_dataset.glob(f'annotations.{ext}')]
        if len(_metadata_files) == 0:
            raise ValueError(f'No metadata.csv or metadata.parquet found in {self.original_dataset}')
        if len(_metadata_files) > 1:
            raise ValueError(f'Found both metadata.csv and metadata.parquet in {self.original_dataset} — ambiguous')
        if len(_label_files) > 1:
            raise ValueError(f'Found both metadata.csv and metadata.parquet in {self.original_dataset} — ambiguous')
        self.original_labels = _label_files[0] if len(_label_files) == 1 else None
        self.original_metadata = _metadata_files[0]
        self.out_dataset = Path(options['reviewed_data_destn'])
        self.out_dataset.mkdir(exist_ok=True, parents=True)
        self.out_labels = self.out_dataset / 'annotations.parquet'
        self.out_metadata = self.out_dataset / 'metadata.parquet'
        self.naming_csv = Path(options['naming_csv'])

### Initialize

In [57]:
paths = FilePaths(use_case)
namer = MiniBirdNamer(paths.naming_csv)
map = FastMap(map_extents=use_case['map_extents'],
              provider =cx.providers.OpenStreetMap.Mapnik) # type: ignore[attr-defined])    #.OpenTopoMap, OpenStreetMap.Mapnik, 
annotation_state = AnnotationState(all_classes=namer.common_names, namer=namer, max_visible=30)

## Check the data tables
Start by loading the metadata dataframe, there should be exactly one row per file

In [58]:
if paths.original_metadata.suffix == '.csv':
    df_meta = pd.read_csv(paths.original_metadata)
else:
    df_meta = pd.read_parquet(paths.original_metadata)

df_meta = df_meta.sort_values(by='filename')  #Ensures all the files from one class folder are presented sequentially
df_meta['secondary_labels'] = df_meta['secondary_labels'].apply(normalize_secondary_labels)
df_meta.head(3)

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
0,20211011_134222_0.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_0.wav,NaN,NaN,NaN
1,20211011_134222_1.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_1.wav,NaN,NaN,NaN
2,20211011_134222_2.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_2.wav,NaN,NaN,NaN


In [59]:
len(df_meta)

6

In [60]:
df_meta.head(3)

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
0,20211011_134222_0.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_0.wav,NaN,NaN,NaN
1,20211011_134222_1.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_1.wav,NaN,NaN,NaN
2,20211011_134222_2.wav,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20211011_134222_2.wav,NaN,NaN,NaN


In [61]:
df_labels = load_labels(paths.original_labels)
df_labels.head()

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Life Stage,Indv ID,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz)


In [62]:
df_naming = pd.read_csv(paths.naming_csv)
df_naming.head(3)

,CommonName,eBird,ScientificName,ExtraName
0,Bellbird,nezbel1,Anthornis melanura,Bellbird
1,Australasian Bittern,ausbit1,Botaurus poiciloptilus,Bittern
2,Bittern,nezbit1,Ixobrychus novaezelandiae,Bittern


## Annotation
* **Every** Bird or Animal Sound to be boxed
* Unknown classes to be labelled *Unknown*
* Calls from the same bird with gaps of greater than 2 seconds should have individual boxes
* Otherwise a single large box should be used
* Spacebar to clear all labels
* Right click to clear the last label
* The t key will attempt to box all patterns matching the contents of the most recent box

In [ ]:
create_class_widgets(annotation_state, n_columns=6, fastmap=map, common_to_ebird=namer.common_to_ebird_dict)

annotator = SpectrogramAnnotator(annotation_state,
                                 common_to_ebird=namer.common_to_ebird_dict,
                                 plot_size = (use_case['display_width'],4),  #Adjust for screen size
                                 f_min=20,
                                 f_max=16000,
                                 zoom_window_height=0.4,
                                 zoom_window_width=5,
                                 min_drag_rows=5,
                                 min_drag_time_s=.1,
                                 min_separation = 2,
                                 similarness_threshold=0.5,
                                 min_freq_hz=300)    ####################Not working yet #########################

session = AnnotationSession(df_meta=df_meta,
                            df_labels=df_labels,
                            new_meta_filepath=paths.out_metadata,
                            new_labels_filepath=paths.out_labels,
                            reviewer = use_case['reviewer'],
                            author = use_case['author'])
controls = AnnotationControls()
annotation_widget = load_current_sample(session, annotator, paths, map)
controls.display()
controls.bind(session, annotator, paths, map)

def on_space_key(event):
    if event.key == ' ':
        controls._on_next_clicked()

annotator.fig.canvas.mpl_connect('key_press_event', on_space_key)

20

## Shortcuts

| Action | Result |
|--------| ---------------- |
| **Left mouse click-drag** |Starts box drawing on click, finishes on release |
| **Left mouse click** |Repeats previous box, but centred on the pointer|
| **Right mouse click** | Moves the zoom box and restarts the playback at that point|
| **u** | Undoes the last box |
| **d** | Deletes all boxes (including originals) |
| **t** | Tries to box any identical patterns from the last un the same frequency limits |
| **b** | Tries to mark calls based on power peaks.  Not recommended, needs improvement |
| **g** | Places a grid of 10 second spacing |
| **g again** | A vertical and horizontal grid |
| **g again** | No grid |

### Progress Checks

In [69]:
session.summary()

{'total_files': 6,
 'finished_files_in_new_meta': 1,
 'total_annotations': 3,
 'done_in_current_session': np.int64(1),
 'pending_in_current_session': np.int64(5),
 'total_minus_pending': np.int64(1)}

In [70]:
marked_times = annotator.get_boxes()
marked_times[-1:]

[]

In [71]:
if paths.out_metadata.exists():
    output_metadata = pd.read_parquet(paths.out_metadata)
    display(output_metadata.tail(10))

,filename,collection,primary_label,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_sr_khz,source_start_s,source_end_s,source_device,models_used
0,20211011_134222_0.wav,<NA>,<NA>,[],<NA>,NaN,NaN,Olly,<NA>,NaT,<NA>,2026-07-23,20211011_134222_0.wav,NaN,NaN,NaN,<NA>,<NA>


In [72]:
if paths.out_metadata.exists():
    display(output_metadata.shape)

(1, 18)

In [68]:
if paths.out_labels.exists():
    output_labeldata = pd.read_parquet(paths.out_labels)
    display(output_labeldata.tail(5))